# 🎬 FastDTW: تجربة عدد الفريمات (10-100) — 40 فيديو

اختبار شامل: 
- بناء بنك خارجي من 4 داتاسِتس (18 حركة، 35 قصاصة)
- تشغيل التجربة عند 10 أعداد فريمات (10, 20, 30...100)
- **رسم 40 فيديو:**
  - 4 فيديوهات × 10 إعدادات = 40 فيديو Raw
  - 4 فيديوهات × 10 إعدادات = 40 فيديو Z-normalized
  - **المجموع: 80 فيديو**

## الخطوة 1: استنساخ الريبو

In [ ]:
!git clone -q -b hmdb51-frame-scale-experiment https://github.com/gradcs2027/Test1.git /kaggle/working/Test1
%cd /kaggle/working/Test1
!git log --oneline -1
!python shared/paths.py

## الخطوة 2: تثبيت المكتبات

In [ ]:
!pip install -q ultralytics
print('✅ ultralytics installed')

## الخطوة 3: توصيل الداتاسِتس

**اضغط + Add Data واختر:**
1. `jizeyong/hmdb51`
2. `jonathannield/cctv-action-recognition-dataset`
3. `jizeyong/charades` (اختياري)
4. `matthewjansen/ucf101-action-recognition`
5. **`gradcs2027/testvid`** ← مهم للفيديوهات الأصلية

## الخطوة 4: بناء البنك الخارجي

In [ ]:
!python fastdtw/build_external_templates.py 2>&1 | tail -50

## الخطوة 5: تشغيل التجربة

In [ ]:
!python fastdtw/experiment_frame_scales.py

## الخطوة 6: تحليل النتائج

In [ ]:
import numpy as np
import pandas as pd

raw = np.load('fastdtw/results/frame_scale_summary.npy', allow_pickle=True)
z   = np.load('fastdtw/results/frame_scale_summary_zscore.npy', allow_pickle=True)

df = pd.DataFrame({
    'فريمات': [r['n_frames'] for r in raw],
    'دقة خام %': [r['accuracy'] for r in raw],
    'دقة Z %': [r['accuracy'] for r in z],
})

print(df.to_string(index=False))

## 🎬 الخطوة 7: رسم 80 فيديو (أهم خطوة)

⏱️ **تحذير:** هذه الخطوة طويلة!
- 10 إعدادات × 4 فيديوهات × 2 نسخة (raw + Z) = 80 فيديو
- الوقت المتوقع: **30-45 دقيقة**
- كل فيديو: 1-5 دقايق

In [ ]:
!python fastdtw/render_all_frame_configs.py

## الخطوة 8: عرض ملخص الفيديوهات

In [ ]:
from pathlib import Path

outputs = Path('outputs')
raw_videos = sorted((outputs / 'raw').glob('*.mp4'))
z_videos = sorted((outputs / 'z_normalized').glob('*.mp4'))

print("="*70)
print(f"📁 الفيديوهات المرسومة")
print("="*70)
print(f"\nRaw (بدون معايرة): {len(raw_videos)} فيديو")
for i, v in enumerate(raw_videos[:5], 1):
    size = v.stat().st_size / 1024 / 1024
    print(f"  {i}. {v.name} ({size:.1f} MB)")
if len(raw_videos) > 5:
    print(f"  ... و {len(raw_videos) - 5} فيديو آخر")

print(f"\nZ-normalized (مع معايرة): {len(z_videos)} فيديو")
for i, v in enumerate(z_videos[:5], 1):
    size = v.stat().st_size / 1024 / 1024
    print(f"  {i}. {v.name} ({size:.1f} MB)")
if len(z_videos) > 5:
    print(f"  ... و {len(z_videos) - 5} فيديو آخر")

print(f"\n✅ الإجمالي: {len(raw_videos) + len(z_videos)} فيديو")
print(f"📁 المسار: {outputs.absolute()}")

## الخطوة 9: تحميل الفيديوهات

**اضغط Files → outputs → raw و z_normalized → Download كل فيديو**

In [ ]:
# لو عايز تضغط الفيديوهات قبل التنزيل:
import zipfile
from pathlib import Path

outputs = Path('outputs')
zip_path = 'videos.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for video in sorted(outputs.rglob('*.mp4')):
        zf.write(video, arcname=video.relative_to(outputs.parent))

size_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f"✅ {zip_path} ({size_mb:.1f} MB)")
print(f"بتقدر تحمّله من Files")

## الملخص النهائي

In [ ]:
print("="*70)
print("📊 النتائج الكاملة")
print("="*70)
print()
print("✅ 35 قصاصة من 4 داتاسِتس")
print("✅ 10 أعداد فريمات (10, 20...100)")
print("✅ 40 نافذة اختبار من vidtest1-4")
print()
print("🎬 80 فيديو مرسوم:")
print("   - 40 فيديو Raw (بدون معايرة)")
print("   - 40 فيديو Z-normalized (مع معايرة)")
print()
print("📁 المسار: outputs/")
print("   ├─ raw/")
print("   │  ├─ 10_vidtest1.mp4")
print("   │  ├─ 10_vidtest2.mp4")
print("   │  └─ ... (40 فيديو)")
print("   └─ z_normalized/")
print("      ├─ 10_vidtest1.mp4")
print("      ├─ 10_vidtest2.mp4")
print("      └─ ... (40 فيديو)")
print()
print("="*70)